# Estudo de Redes Neurais Recorrentes — Dataset IARA
### Notebook 25: Avaliação Completa do BiGRU com Atenção Temporal (MEL, LOFAR, DEMON, HHT e Fusões Multimodais)

Neste notebook, investigamos a arquitetura **Bidirectional Gated Recurrent Unit (BiGRU) com Mecanismo de Atenção Temporal** no problema de classificação acústica submarina passiva (*Underwater Acoustic Target Recognition* - UATR) sobre o dataset **IARA** (*Silva et al.*, IEEE Access 2025).

### Objetivos da Investigação:
1. **Avaliação Temporal vs. Estática:** Verificar se a modelagem explícita da dinâmica temporal de sequências de $L=15$ frames supera os classificadores estáticos (SVM Nyström, MLP e CNN). Os frames dos caches usados aqui têm passo de **$5{,}12\text{ s}$** (média das features em blocos de 5,12 s), logo cada sequência cobre **$\approx 76{,}8\text{ s}$** de áudio, com deslocamento de 5 frames ($\approx 25{,}6\text{ s}$). Gravações com menos de 15 frames viram uma única sequência completada com zeros. Cada arquivo de teste gera de 2 a 9 sequências (média $7{,}38$), e a predição do arquivo é o voto majoritário das sequências.
2. **Comparação de Representações Espectrais e de Envoltória:**
   - **MEL (256-D)** e **LOFAR (1024-D)**: caches `mel5s` / `lofar5s`, com resolução temporal **10× menor** que as features oficiais de 0,512 s do artigo. Os resultados não são, portanto, diretamente comparáveis aos da Tabela 7 do artigo;
   - **DEMON (253-D):** demodulação de envoltória (passagem de pás do hélice);
   - **HHT Pura (256-D):** espectro marginal de Hilbert-Huang (EMD + Transformada de Hilbert);
   - **Fusões:** `MEL + DEMON` (509-D), `MEL + HHT` (512-D), `LOFAR + DEMON` (1277-D) e `LOFAR + HHT` (1280-D), por concatenação frame a frame (truncadas ao menor número de frames).
3. **Análise de Escopo:** **4 Classes** (`SMALL`, `MEDIUM`, `LARGE`, `BACKGROUND`; 639 arquivos de teste por fold) e **3 Classes** (modelo **retreinado** sem `BACKGROUND`; 541 arquivos de teste por fold).

**Protocolo:** validação cruzada $5\times2\text{cv}$ oficial (10 folds, os mesmos arquivos de `results/trainings/tests/*/eval/fold_k`), com particionamento por **gravação de áudio** (`exclusive_header='ID'`, `src/iara/ml/experiment.py`): janelas de uma mesma gravação nunca se misturam entre treino e teste, **mas o mesmo navio pode aparecer em treino e teste** (não há isolamento por `Ship ID`).

**Limitações de treino (declaradas):** não há conjunto de validação nem *early stopping*. Os modelos treinam por um número **fixo** de épocas, e os 214 arquivos de validação de cada fold oficial são descartados (não entram nem no treino). Os scripts não fixam semente (`torch.manual_seed` ausente), então uma nova execução não reproduz os números exatamente.

In [ ]:
import os
import sys
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('notebooks'))
sys.path.insert(0, os.path.abspath('src'))  # módulos movidos para notebooks/src
sys.path.insert(0, os.path.abspath('notebooks/src'))
import iara_results as R

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 120
})

def crnn_fold_metrics(name="crnn2d_multiclass_lofar_s10_fpool32_h128_lr0.0002"):
    """A CRNN 2D grava <exp>_test.csv (sem '_multiclass'); aplica as mesmas convenções de iara_results:
    voto por arquivo com Counter.most_common, SP, ACC balanceada, desvio ddof=0."""
    d = os.path.join(R.ROOT, "results", "trainings", "tests", name)
    rows = []
    for k in range(10):
        df = pd.read_csv(os.path.join(d, "eval", f"fold_{k}", f"{name}_test.csv"), usecols=["File", "Target", "Prediction"])
        y, p = [], []
        for fid, g in df.groupby("File", sort=True):
            y.append(Counter(g["Target"]).most_common(1)[0][0])
            p.append(Counter(g["Prediction"].tolist()).most_common(1)[0][0])
        rows.append({"fold": k, **R._metrics(np.array(y), np.array(p))})
    return pd.DataFrame(rows).set_index("fold")


def ms(series):
    return series.mean(), series.std(ddof=0)

print('Ambiente configurado. Resultados lidos de', R.ROOT)

## 1. Configuração dos Dois Scripts (lida do código e de `config.json`)

Há **duas arquiteturas diferentes**, uma por script. Toda comparação MEL × LOFAR neste notebook mistura o efeito da representação com o efeito da arquitetura.

| Item | `eval_bigru_multiclass_5x2cv.py` (DEMON, HHT, MEL, MEL+DEMON, MEL+HHT) | `eval_bigru_lofar_multiclass_5x2cv.py` (LOFAR, LOFAR+DEMON, LOFAR+HHT) |
| :--- | :--- | :--- |
| GRU | 2 camadas, bidirecional, `hidden_size=64` (saída 128-D), dropout 0,3 entre camadas | 2 camadas, bidirecional, `hidden_size=128` (saída 256-D), dropout 0,3 entre camadas |
| Atenção | Linear(128→64) → tanh → Linear(→1), softmax no tempo | Linear(256→128) → tanh → Linear(→1), softmax no tempo |
| Classificador | **BatchNorm1d**(128) → Dropout → Linear(→**32**) → ReLU → **BatchNorm1d**(32) → Dropout → Linear | Linear(→**64**) → ReLU → Dropout → Linear (**sem BatchNorm**) |
| Épocas (fixas) | 20 | 15 |
| Pesos de classe | `compute_class_weight('balanced')` = $n/(K\cdot n_c)$ | $1/n_c$ normalizado para somar $K$ |
| `drop_last` no treino | `False` | `True` |
| Comum | AdamW (lr=0.0003, wd=1e-3), CosineAnnealingLR, *clip* de gradiente 1,0, lote 128, `StandardScaler` ajustado nas janelas de treino, $L=15$, passo 5 | idem |

**Agregação por arquivo:** os scripts usam `argmax(bincount)` (empate → menor índice de classe). As métricas abaixo são **recalculadas** a partir das predições por sequência gravadas em disco com a convenção do pipeline `iara` (`Counter.most_common`, empate → classe da primeira sequência), por isso podem diferir ligeiramente dos valores impressos pelos scripts. **ACC = acurácia balanceada** (média dos recalls), como a coluna ACC da Tabela 7 do artigo; desvio-padrão populacional sobre os 10 folds.

## 2. Tabela Consolidada de Resultados (10 Folds $5\times2\text{cv}$)

In [ ]:
BIGRU = [
    ('DEMON', 'bigru_att_demon_h64_s15-5_e20', 253),
    ('HHT', 'bigru_att_hht_h64_s15-5_e20', 256),
    ('MEL (mel5s)', 'bigru_att_mel5s_h64_s15-5_e20', 256),
    ('FUSÃO MEL + DEMON', 'bigru_att_mel5s-demon_h64_s15-5_e20', 509),
    ('FUSÃO MEL + HHT', 'bigru_att_mel5s-hht_h64_s15-5_e20', 512),
    ('LOFAR (lofar5s)', 'bigru_att_lofar5s_h128_s15-5_e15', 1024),
    ('FUSÃO LOFAR + DEMON', 'bigru_att_lofar5s-demon_h128_s15-5_e15', 1277),
    ('FUSÃO LOFAR + HHT', 'bigru_att_lofar5s-hht_h128_s15-5_e15', 1280)
]
BASELINES = [
    ('MLP LOFAR (≈ artigo)', 'mlp_lofar'),
    ('MLP MEL (≈ artigo)', 'mlp'),
    ('CNN MEL (artigo)', 'cnn_mel_artigo'),
    ('SVM Nyström m=4000 MEL', 'svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0'),
    ('SVM Nyström m=6000 LOFAR', 'svm_nystroem_6000_lofar_elasticnet_l1r0.15')
]

def linha(escopo, modelo, rep, dim, s, exp):
    return {'Escopo': escopo, 'Modelo': modelo, 'Representacao': rep, 'Dim': dim,
            'SP (%)': s['SP'][0], 'SP_std': s['SP'][1], 'Bal Acc (%)': s['ACC'][0], 'Bal Acc_std': s['ACC'][1],
            'Rec SMALL (%)': s['R_SMALL'][0], 'Rec MED (%)': s['R_MEDIUM'][0], 'Rec LARGE (%)': s['R_LARGE'][0],
            'Rec BG (%)': s['R_BACKGROUND'][0] if 'R_BACKGROUND' in s else np.nan, 'Experimento': exp}

rows = []
for escopo, suf in [('4 Classes', '4c'), ('3 Classes', '3c')]:
    for rep, tag, dim in BIGRU:
        exp = f'{tag}_{suf}'
        rows.append(linha(escopo, 'BiGRU + Atenção', rep, dim, R.summary(exp, suf), exp))
for modelo, exp in BASELINES:
    rows.append(linha('4 Classes', modelo, '', np.nan, R.summary(exp), exp))
crnn = crnn_fold_metrics()
rows.append(linha('4 Classes', 'CRNN 2D LOFAR (re-execução local)', 'LOFAR', 1024,
                  {c: ms(crnn[c]) for c in crnn.columns}, 'crnn2d_multiclass_lofar_s10_fpool32_h128_lr0.0002'))
for (mod, rep), v in R.ARTICLE_TABLE7.items():
    if mod in ('MLP', 'CNN'):
        rows.append({'Escopo': '4 Classes', 'Modelo': f'{mod} (Tabela 7 do artigo)', 'Representacao': rep, 'Dim': np.nan,
                     'SP (%)': v['SP'][0], 'SP_std': v['SP'][1], 'Bal Acc (%)': v['ACC'][0], 'Bal Acc_std': v['ACC'][1],
                     'Rec SMALL (%)': np.nan, 'Rec MED (%)': np.nan, 'Rec LARGE (%)': np.nan, 'Rec BG (%)': np.nan,
                     'Experimento': 'artigo (features de 0,512 s)'})

df_results = pd.DataFrame(rows)
display(df_results.round(2))

## 3. Teste Estatístico: F-test $5\times2\text{cv}$ contra a MLP LOFAR

O artigo compara modelos pelo F-test $5\times2\text{cv}$ combinado (Alpaydin). Abaixo, cada BiGRU (4 classes) é testado contra a **MLP LOFAR re-executada localmente** (`mlp_lofar`, SP $66{,}24 \pm 1{,}78\%$, mesmos folds). Contra o **artigo** não é possível testar (não há SP por fold publicado); a MLP LOFAR do artigo tem SP $66{,}51 \pm 1{,}39\%$ (Tabela 7), com features de 0,512 s.

In [ ]:
ft = []
for rep, tag, dim in BIGRU:
    exp = f'{tag}_4c'
    s = R.summary(exp)
    F, p = R.f_test_5x2(exp, 'mlp_lofar')
    ft.append({'BiGRU (4c)': rep, 'SP (%)': R.fmt(s['SP']), 'Δ SP vs MLP LOFAR local (p.p.)': s['SP'][0] - R.summary('mlp_lofar')['SP'][0],
               'F': F, 'p': p, 'Diferença significativa (p<0,05)?': 'sim' if p < 0.05 else 'não',
               'Δ SP vs MLP LOFAR do artigo (p.p.)': s['SP'][0] - R.ARTICLE_TABLE7[('MLP', 'LOFAR')]['SP'][0]})
F, p = R.f_test_5x2(crnn['SP'].values, 'mlp_lofar')
ft.append({'BiGRU (4c)': 'CRNN 2D LOFAR (referência)', 'SP (%)': R.fmt(ms(crnn['SP'])),
           'Δ SP vs MLP LOFAR local (p.p.)': crnn['SP'].mean() - R.summary('mlp_lofar')['SP'][0], 'F': F, 'p': p,
           'Diferença significativa (p<0,05)?': 'sim' if p < 0.05 else 'não',
           'Δ SP vs MLP LOFAR do artigo (p.p.)': crnn['SP'].mean() - R.ARTICLE_TABLE7[('MLP', 'LOFAR')]['SP'][0]})
display(pd.DataFrame(ft).round(3))

**Leitura:** em 4 classes, são **significativamente piores** que a MLP LOFAR local: DEMON (p = $0{,}006$), HHT (p = $0{,}017$). As demais configurações ficam **abaixo em média, mas sem diferença significativa** pelo F-test: MEL (mel5s) (p = $0{,}084$), FUSÃO MEL + DEMON (p = $0{,}480$), FUSÃO MEL + HHT (p = $0{,}262$), LOFAR (lofar5s) (p = $0{,}070$), FUSÃO LOFAR + DEMON (p = $0{,}293$), FUSÃO LOFAR + HHT (p = $0{,}384$). O melhor BiGRU em 4 classes é **FUSÃO LOFAR + DEMON** (SP $64{,}04 \pm 1{,}42\%$), $2{,}20$ p.p. abaixo da MLP LOFAR local e $2{,}47$ p.p. abaixo da MLP LOFAR do artigo.

## 4. Gráfico Comparativo: Índice SP (%) e Acurácia Balanceada por Representação

SP com barras de desvio-padrão entre os 10 folds, para as 8 configurações do BiGRU em **4 Classes** e **3 Classes**. As linhas de referência (só no painel de 4 classes) são a **MLP LOFAR local** ($66{,}24\%$), a **MLP LOFAR do artigo** ($66{,}51\%$) e o **SVM MEL m=4000 local** ($63{,}78\%$). Os rótulos mostram SP e, entre parênteses, a acurácia balanceada.

In [ ]:
df_4c = df_results[(df_results['Escopo'] == '4 Classes') & (df_results['Modelo'] == 'BiGRU + Atenção')]
df_3c = df_results[(df_results['Escopo'] == '3 Classes') & (df_results['Modelo'] == 'BiGRU + Atenção')]
sp_ref = R.summary('mlp_lofar')['SP'][0]
sp_art = R.ARTICLE_TABLE7[('MLP', 'LOFAR')]['SP'][0]
sp_svm = R.summary('svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0')['SP'][0]

reps_labels = ['DEMON', 'HHT', 'MEL', 'MEL+DEM', 'MEL+HHT', 'LOFAR', 'LOF+DEM', 'LOF+HHT']
colors = ['#95a5a6', '#e67e22', '#3498db', '#9b59b6', '#2ecc71', '#34495e', '#e74c3c', '#1abc9c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
bars1 = ax1.bar(np.arange(len(reps_labels)), df_4c['SP (%)'], yerr=df_4c['SP_std'], capsize=5, color=colors, alpha=0.85, edgecolor='black', lw=1.2)
ax1.axhline(sp_svm, color='blue', linestyle='--', label=f'SVM MEL m=4000 local ({sp_svm:.1f}%)', lw=1.6)
ax1.axhline(sp_ref, color='red', linestyle='--', label=f'MLP LOFAR local ({sp_ref:.1f}%)', lw=1.6)
ax1.axhline(sp_art, color='darkred', linestyle=':', label=f'MLP LOFAR artigo ({sp_art:.1f}%)', lw=1.6)
ax1.set_title("BiGRU com Atenção — 4 Classes (SP Global)", fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel("Standard Product (SP) %", fontsize=11, fontweight='bold')
ax1.set_xticks(np.arange(len(reps_labels)))
ax1.set_xticklabels(reps_labels, fontsize=10, fontweight='bold', rotation=20)
ax1.set_ylim(45, 72)
ax1.legend(loc='lower left', frameon=True)
for bar, sp, ba in zip(bars1, df_4c['SP (%)'], df_4c['Bal Acc (%)']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.6, f"{sp:.1f}%\n({ba:.1f}%)",
             ha='center', va='bottom', fontsize=8.5, fontweight='bold')

bars2 = ax2.bar(np.arange(len(reps_labels)), df_3c['SP (%)'], yerr=df_3c['SP_std'], capsize=5, color=colors, alpha=0.85, edgecolor='black', lw=1.2)
ax2.set_title("BiGRU com Atenção — 3 Classes (modelo retreinado sem BACKGROUND)", fontsize=13, fontweight='bold', pad=12)
ax2.set_ylabel("Standard Product (SP) %", fontsize=11, fontweight='bold')
ax2.set_xticks(np.arange(len(reps_labels)))
ax2.set_xticklabels(reps_labels, fontsize=10, fontweight='bold', rotation=20)
ax2.set_ylim(45, 70)
for bar, sp, ba in zip(bars2, df_3c['SP (%)'], df_3c['Bal Acc (%)']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.6, f"{sp:.1f}%\n({ba:.1f}%)",
             ha='center', va='bottom', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Recalls por Classe: o Alvo Crítico SMALL

A principal dificuldade no IARA é reconhecer embarcações de pequeno porte (**SMALL**). O gráfico mostra os recalls médios por classe de cada configuração:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
w = 0.25
x = np.arange(len(reps_labels))
for ax, d, titulo in [(ax1, df_4c, "Recalls por Classe em 4 Classes"), (ax2, df_3c, "Recalls por Classe em 3 Classes (retreinado sem BACKGROUND)")]:
    ax.bar(x - w, d['Rec SMALL (%)'], width=w, label='SMALL', color='#3498db', edgecolor='black', alpha=0.85)
    ax.bar(x, d['Rec MED (%)'], width=w, label='MEDIUM', color='#f39c12', edgecolor='black', alpha=0.85)
    ax.bar(x + w, d['Rec LARGE (%)'], width=w, label='LARGE', color='#2ecc71', edgecolor='black', alpha=0.85)
    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=12)
    ax.set_ylabel("Recall (%)", fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(reps_labels, fontsize=9.5, fontweight='bold', rotation=20)
    ax.set_ylim(20, 85)
    ax.legend(loc='upper left', frameon=True)
    for i in range(len(reps_labels)):
        for dx, col in [(-w, 'Rec SMALL (%)'), (0, 'Rec MED (%)'), (w, 'Rec LARGE (%)')]:
            ax.text(x[i] + dx, d[col].iloc[i] + 1, f"{d[col].iloc[i]:.0f}%", ha='center', fontsize=7.5, fontweight='bold')
plt.tight_layout()
plt.show()

# F-test 5x2cv sobre o recall de SMALL / MEDIUM (por fold)
pares = [('LOFAR vs MEL (SMALL)', 'bigru_att_lofar5s_h128_s15-5_e15_4c', 'bigru_att_mel5s_h64_s15-5_e20_4c', 'R_SMALL'),
         ('LOFAR+DEMON vs LOFAR (SMALL)', 'bigru_att_lofar5s-demon_h128_s15-5_e15_4c', 'bigru_att_lofar5s_h128_s15-5_e15_4c', 'R_SMALL'),
         ('LOFAR+DEMON vs MLP LOFAR local (SMALL)', 'bigru_att_lofar5s-demon_h128_s15-5_e15_4c', 'mlp_lofar', 'R_SMALL'),
         ('LOFAR vs MEL (MEDIUM)', 'bigru_att_lofar5s_h128_s15-5_e15_4c', 'bigru_att_mel5s_h64_s15-5_e20_4c', 'R_MEDIUM')]
out = []
for nome, a, b, m in pares:
    F, p = R.f_test_5x2(a, b, metric=m)
    out.append({'Comparação': nome, 'A (%)': R.summary(a)[m][0], 'B (%)': R.summary(b)[m][0], 'F': F, 'p': p})
display(pd.DataFrame(out).round(3))

## 6. Discussão Científica e Conclusões

### A. Existe um "teto estrutural" dos modelos temporais?
- A versão anterior deste notebook afirmava um teto de ~62% para BiGRU/CRNN. Com as predições re-executadas, o melhor BiGRU em 4 classes é **FUSÃO LOFAR + DEMON** com SP $64{,}04 \pm 1{,}42\%$, e em 3 classes **FUSÃO MEL + HHT** com $62{,}69 \pm 1{,}55\%$.
- Contra a MLP LOFAR local ($66{,}24 \pm 1{,}78\%$), só DEMON (p = $0{,}006$), HHT (p = $0{,}017$) são significativamente piores. As outras configurações são **estatisticamente equivalentes** pelo F-test $5\times2\text{cv}$ (p ≥ $0{,}070$), embora fiquem abaixo em média. Não há, portanto, evidência de um "teto" específico dos modelos recorrentes; também não há evidência de que eles superem os estáticos.
- As fusões não trazem ganho significativo sobre a representação base: MEL+DEMON × MEL p = $0{,}622$; MEL+HHT × MEL p = $0{,}313$; LOFAR+DEMON × LOFAR p = $0{,}057$.
- A CRNN 2D LOFAR re-executada (`results/trainings/tests/crnn2d_multiclass_lofar_s10_fpool32_h128_lr0.0002`, recalculada com a convenção `iara`; o `metrics_summary.csv` gravado pelo script de treino dá SP $60{,}47\%$) tem SP $60{,}51 \pm 2{,}19\%$ (recalls SMALL/MEDIUM/LARGE = $36{,}7$ / $68{,}8$ / $62{,}9\%$); contra o BiGRU LOFAR ($62{,}22\%$), p = $0{,}570$.
- **Ressalvas que limitam a interpretação:** (i) as features MEL/LOFAR do BiGRU têm resolução de 5,12 s, 10× menor que a do artigo, então a comparação com a Tabela 7 mistura modelo e representação; (ii) épocas fixas, sem validação nem *early stopping*; (iii) as variantes MEL e LOFAR usam arquiteturas diferentes (Seção 1); (iv) o protocolo separa gravações, não navios. Hipóteses como "contaminação dos estados ocultos por variações de SNR" ou "cadência temporal específica dos navios de treino" **não foram testadas** neste estudo.

### B. Recall de SMALL
- O BiGRU MEL tem recall de SMALL de $45{,}3\%$ e o BiGRU LOFAR, $55{,}8\%$ (F-test sobre o recall de SMALL: p = $0{,}235$, diferença não significativa). Mas a troca de entrada também troca a arquitetura, e o recall de MEDIUM cai de $62{,}8\%$ para $51{,}2\%$ (p = $0{,}016$, diferença significativa).
- A fusão LOFAR + DEMON chega a $57{,}7\%$ em 4 classes (vs. LOFAR puro, p = $0{,}600$, diferença não significativa) e $58{,}5\%$ em 3 classes. Contra a MLP LOFAR local ($46{,}3\%$), p = $0{,}105$ (diferença não significativa).
- Entre as redes neurais desta tabela, o maior recall de SMALL em 4 classes é o de **BiGRU FUSÃO LOFAR + DEMON (4c)** ($57{,}7\%$). Isso vale só para os modelos listados aqui; não é uma afirmação sobre "todas as redes do estudo".

### C. Recomendação Prática
Os classificadores estáticos (MLP e SVM) continuam sendo a opção mais simples: SP estatisticamente equivalente ao dos melhores BiGRU (a MLP LOFAR local ainda é maior em média), menor custo e sem dependência de sequências de $\approx 77\text{ s}$. O BiGRU com LOFAR (+ DEMON) mostra um perfil de recalls diferente (mais SMALL, menos MEDIUM; só a queda de MEDIUM é significativa nos testes acima). Isso o torna um **candidato** a especialista de SMALL, mas a hipótese só se sustenta depois de um teste num sistema realizável (por exemplo, uma cascata avaliada no 5x2cv), e não apenas comparando recalls.